In [34]:
import sys
sys.path.append('..')

from utils.spark_session import get_spark_session
from utils.parquet_io import read_parquet, write_partitioned_parquet, write_parquet
from pyspark.sql.functions import col, dayofweek, hour, when, round, broadcast

# Initialize Spark session
spark = get_spark_session()
print('session created')

session created


In [35]:
# Read parquet files
taxi01_df = read_parquet(spark, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\03_feature_data\\taxi01")
taxi02_df = read_parquet(spark, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\03_feature_data\\taxi02")
zone_df = read_parquet(spark, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\03_feature_data\\zone")

In [36]:
zone_cache = zone_df.cache()

In [61]:
# Broadcast join

taxi01_zone_df = taxi01_df.join(
    broadcast(zone_cache),
    taxi01_df.PULocationID == zone_cache.LocationID,
    "left"
)

taxi02_zone_df = taxi02_df.join(
    broadcast(zone_cache),
    taxi02_df.PULocationID == zone_cache.LocationID,
    "left"
)


In [62]:
taxi01_zone_df.groupBy("VendorID").count().show()

+--------+-------+
|VendorID|  count|
+--------+-------+
|       1|4397921|
|       2|5308357|
+--------+-------+



In [ ]:
#taxi01_zone_df = taxi01_zone_df.repartition(16) 


In [ ]:
#taxi01_zone_df = taxi01_zone_df.coalesce(8) 


In [65]:
# Save join data
taxi01_zone_df = taxi01_zone_df.repartition(16, "VendorID") 

write_partitioned_parquet(taxi01_zone_df, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\07_optimized_data\\taxi01", "VendorID")

In [64]:
write_partitioned_parquet(taxi02_zone_df, "C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\07_optimized_data\\taxi02", "is_weekend")

In [67]:
print("Data saved successfully!")

spark.stop()

Data saved successfully!
